<a href="https://colab.research.google.com/github/Pradeep1694/Capstone-Project/blob/main/Capstone_Project_Part_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [90]:
!pip install requests jsonschema joblib

In [92]:
import os
import json
import re
import joblib
import requests
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
BASE_URL = "https://openrouter.ai/api/v1"

def call_llm(system_prompt,user_prompt,temperature=0,max_token=512):
	url = "https://openrouter.ai/api/v1/chat/completions"
	header = {"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
	          "Content-Type": "application/json"
	          }
	payload = {"model": "openai/gpt-4o-mini",
	           "messages":[{"role": "system", "content": system_prompt},
	                      {"role": "user", "content": user_prompt}],
	           "temperature": temperature,
	           "max_tokens": max_token
	           }
	response = requests.post(url,headers=header,json=payload)
	if response.status_code!=200:
	  print(response.status_code)
	  print(response.text)
	  return None

	return response.json()["choices"][0]["message"]["content"]

system_prompt="Reply with only one word."

user_prompt="Hello"

print(call_llm(system_prompt, user_prompt))


Hi!


In [93]:
import pandas as pd

df = pd.read_csv("cleaned_data.csv")

feature=df[[
    "actual_price",
    "discount_percentage",
    "rating",
    "rating_count",
    "category"
]]

record1 = feature.iloc[0]
record2 = feature.iloc[1]
record3 = feature.iloc[2]

print(record1)
print(record2)
print(record3)

actual_price                                                      1099.0
discount_percentage                                                 64.0
rating                                                               4.2
rating_count                                                     24269.0
category               Computers&Accessories|Accessories&Peripherals|...
Name: 0, dtype: object
actual_price                                                       349.0
discount_percentage                                                 43.0
rating                                                               4.0
rating_count                                                     43994.0
category               Computers&Accessories|Accessories&Peripherals|...
Name: 1, dtype: object
actual_price                                                      1899.0
discount_percentage                                                 90.0
rating                                                               3.9
ratin

In [94]:
import joblib
model = joblib.load("best_model.pk1")

print("model loaded successfully")

x_test =pd.DataFrame(columns=model.feature_names_in_)

x_test.loc[0] = record1
x_test.loc[1] = record2
x_test.loc[2] = record3

print(x_test)

predictions = model.predict(x_test)

probabilities = model.predict_proba(x_test)

for i in range(len(predictions)):
  print(f"\nRecord{i+1}")
  print(f"Prediction {i+1}: {predictions[i]}")
  print("Probability:", probabilities[i])



model loaded successfully
  actual_price discount_percentage rating rating_count  \
0       1099.0                64.0    4.2      24269.0   
1        349.0                43.0    4.0      43994.0   
2       1899.0                90.0    3.9       7928.0   

  category_Computers&Accessories|Accessories&Peripherals|Adapters|USBtoUSBAdapters  \
0                                                NaN                                 
1                                                NaN                                 
2                                                NaN                                 

  category_Computers&Accessories|Accessories&Peripherals|Audio&VideoAccessories|PCHeadsets  \
0                                                NaN                                         
1                                                NaN                                         
2                                                NaN                                         

  category_Computer

In [95]:
system_prompt = """
You are an AI prediction explanation assistant.

Explain the machine learning prediction in simple language.

Return ONLY valid JSON.

{
 "prediction_label":"",
 "confidence_level":"",
 "top_reason":"",
 "second_reason":"",
 "next_step":""
}

Do not include markdown.
"""
response =call_llm(system_prompt,user_prompt,temperature=0.0)
print(response)

{
 "prediction_label":"greeting",
 "confidence_level":"high",
 "top_reason":"The input is a common way to initiate a conversation.",
 "second_reason":"It is a simple and direct expression of acknowledgment.",
 "next_step":"Respond with a friendly reply."
}


In [96]:
user_prompt = f"""
Feature Values:

Actual Price: {record1['actual_price']}
Discount Percentage: {record1['discount_percentage']}
Rating: {record1['rating']}
Rating Count: {record1['rating_count']}
Category: {record1['category']}

Predicted Class: {predictions[0]}

Prediction Probability: {max(probabilities[0]):.2f}

Explain why the model made this prediction.

Return JSON only.
"""
print(user_prompt)


Feature Values:

Actual Price: 1099.0
Discount Percentage: 64.0
Rating: 4.2
Rating Count: 24269.0
Category: Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables

Predicted Class: 0

Prediction Probability: 0.98

Explain why the model made this prediction.

Return JSON only.



In [100]:


os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
BASE_URL = "https://openrouter.ai/api/v1"

def call_llm(system_prompt,user_prompt,temperature=0,max_token=300):
	url = "https://openrouter.ai/api/v1/chat/completions"
	header = {"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
	          "Content-Type": "application/json"
	          }
	payload = {"model": "openai/gpt-4o-mini",
	           "messages":[{"role": "system", "content": system_prompt},
	                      {"role": "user", "content": user_prompt}],
	           "temperature": temperature,
	           "max_tokens": max_token
	           }
	response = requests.post(url,headers=header,json=payload)

	return response.json()["choices"][0]["message"]["content"]

response = (call_llm(system_prompt,user_prompt))
print(response)


{
 "prediction_label":"Not Recommended",
 "confidence_level":"0.98",
 "top_reason":"The high discount percentage of 64% suggests that the product may not be of high quality or is being sold at a significantly reduced price, which often indicates lower value.",
 "second_reason":"Despite a decent rating of 4.2, the large number of ratings (24269) indicates that many customers have purchased this product, and the overall sentiment may not be strong enough to outweigh the concerns raised by the discount.",
 "next_step":"Consider looking for products with better ratings and lower discount percentages for better quality."
}


In [101]:
from jsonschema import validate, ValidationError

# Define the JSON schema

schema = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string"},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ]
}

from jsonschema import validate, ValidationError
import json

try:
  output = json.loads(response)

  validate(instance=output,schema=schema)

  print("JSON Validation passed")
  print(output)

except ValidationError as e:
  print("JSON validation Failed")
  print(e)

except json.JSONDecodeError:
  print("Invalid JSON retuened by the LLM")


JSON Validation passed
{'prediction_label': 'Not Recommended', 'confidence_level': '0.98', 'top_reason': 'The high discount percentage of 64% suggests that the product may not be of high quality or is being sold at a significantly reduced price, which often indicates lower value.', 'second_reason': 'Despite a decent rating of 4.2, the large number of ratings (24269) indicates that many customers have purchased this product, and the overall sentiment may not be strong enough to outweigh the concerns raised by the discount.', 'next_step': 'Consider looking for products with better ratings and lower discount percentages for better quality.'}


In [102]:
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))
    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )

if has_pii(user_prompt):
  print("Input blocked: PII detected")
else:
  response = call_llm(system_prompt,user_prompt)
  print(response)

{
 "prediction_label":"Not Recommended",
 "confidence_level":"0.98",
 "top_reason":"The high discount percentage of 64% suggests that the product may not be of high quality or is being sold at a significantly reduced price, which often indicates lower value.",
 "second_reason":"Despite a decent rating of 4.2, the large number of ratings (24269) indicates that many customers have reviewed the product, and the overall sentiment may not be strong enough to outweigh the concerns raised by the discount.",
 "next_step":"Consider looking for products with better ratings and lower discount percentages for higher quality."
}
